In [104]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

In [105]:
pd.set_option('display.max_columns',None)

In [106]:
url = 'https://raw.githubusercontent.com/ichiP245/my-next-soderling/refs/heads/main/Archivos/df_tenis_columns_OK.csv'

In [107]:
df = pd.read_csv(url)

In [108]:
df['Fecha'] = pd.to_datetime(df['Fecha'], format='%Y-%m-%d')

In [109]:
df.columns

Index(['Unnamed: 0', 'ATP', 'Location', 'Tournament', 'Date', 'Series',
       'Court', 'Surface', 'Round', 'Best of', 'Winner', 'Loser', 'WRank',
       'LRank', 'WPts', 'LPts', 'W1', 'L1', 'W2', 'L2', 'W3', 'L3', 'W4', 'L4',
       'W5', 'L5', 'Wsets', 'Lsets', 'Comment', 'B365W', 'B365L', 'MaxW',
       'MaxL', 'AvgW', 'AvgL', 'Fecha', 'playerA', 'playerB', 'rankA', 'rankB',
       'PtsA', 'PtsB', 'B365A', 'B365B', 'MaxA', 'MaxB', 'AvgA', 'AvgB', 'A1',
       'B1', 'A2', 'B2', 'A3', 'B3', 'A4', 'B4', 'A5', 'B5', 'setsA', 'setsB',
       'target'],
      dtype='object')

In [110]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 16061 entries, 0 to 16060
Data columns (total 61 columns):
 #   Column      Non-Null Count  Dtype         
---  ------      --------------  -----         
 0   Unnamed: 0  16061 non-null  int64         
 1   ATP         16061 non-null  float64       
 2   Location    16061 non-null  object        
 3   Tournament  16061 non-null  object        
 4   Date        16061 non-null  object        
 5   Series      16061 non-null  object        
 6   Court       16061 non-null  object        
 7   Surface     16061 non-null  object        
 8   Round       16061 non-null  object        
 9   Best of     16061 non-null  float64       
 10  Winner      16061 non-null  object        
 11  Loser       16061 non-null  object        
 12  WRank       16061 non-null  float64       
 13  LRank       16061 non-null  float64       
 14  WPts        16061 non-null  float64       
 15  LPts        16061 non-null  float64       
 16  W1          16061 non-

## Creacion de Variables

Hacemos una columna con cantidad total de sets de ese partido

In [111]:
df['setsPartido'] = df['setsA'] + df['setsB']

Implied Probabilities de cada campo que tiene cuotas (odds)

In [112]:
def implied_prob_adjusted(odds_A, odds_B):
    """Pasamos odds decimales en probabilidades ajustadas eliminando el margen del bookmaker."""
    inv_A = 1 / odds_A
    inv_B = 1 / odds_B

    total = inv_A + inv_B

    prob_A = inv_A / total
    prob_B = inv_B / total

    return prob_A, prob_B

In [113]:
probA = 1 / df['B365A']
probB = 1 / df['B365B']
df['B365BookmakersMargin'] = (probA + probB - 1)

df['B365ProbA'], df['B365ProbB'] = implied_prob_adjusted(df['B365A'], df['B365B'])
df['ProbAvgA'], df['ProbAvgB'] = implied_prob_adjusted(df['AvgA'], df['AvgB'])
df['ProbMaxA'], df['ProbMaxB'] = implied_prob_adjusted(df['MaxA'], df['MaxB'])

df['market_uncertainty'] = np.abs(df['ProbAvgA'] - 0.5) # Mas bajo, mas parejo el partido

# Vemos la diferencia entre las probabilidades mas optimistas y las del consenso general del mercado
df['SpreadOddsA'] = df['ProbMaxA'] - df['ProbAvgA']
df['SpreadOddsB'] = df['ProbAvgB'] - df['ProbMaxB']

# Diferencia promedio de probabilidades
df['AvgOddsDiff'] = (df['ProbAvgA'] - df['ProbAvgB'])

# Probabilidades con un logaritmo aplicado, recomendado por ChatGPT
df['logit_oddsA'] = np.log(df['ProbAvgA'] / (1 - df['ProbAvgA']))
df['logit_oddsB'] = np.log(df['ProbAvgB'] / (1 - df['ProbAvgB']))

Ranking Difference, ATP Points Difference

In [114]:
df['rankDiff'] = df['rankA'] - df['rankB']
df['ptsDiff'] = df['PtsA'] - df['PtsB']

Porcentaje de partidos ganados en los últimos 5, 10 o 20 partidos

In [115]:
jug = pd.concat([df['playerA'], df['playerB']], axis=0).value_counts()
print(f'Total: {jug.shape[0]} jugadores')
print(f'Con mas de 20 partidos: {jug[jug>20].shape[0]} jugadores')
print(f'Con mas de 10 partidos: {jug[jug>10].shape[0]} jugadores')
print(f'Con mas de 5 partidos: {jug[jug>5].shape[0]} jugadores')

Total: 613 jugadores
Con mas de 20 partidos: 264 jugadores
Con mas de 10 partidos: 325 jugadores
Con mas de 5 partidos: 385 jugadores


In [116]:
# Hacemos un df con Jugadores A y otro con Jugadores B y los concatenamos uno abajo del otro
a = df[['Fecha','playerA','target', 'setsPartido', 'Unnamed: 0']].rename(columns={'playerA':'Player','target':'win'})
b = df[['Fecha','playerB','target', 'setsPartido', 'Unnamed: 0']].rename(columns={'playerB':'Player','target':'win'})
b['win'] = 1 - b['win']

matches_long = pd.concat([a, b], ignore_index=True).sort_values("Fecha").reset_index(drop=True)

In [117]:
# Calculamos winrate de los ultimos 5 partidos
matches_long['winrate_5'] = (matches_long.groupby('Player')['win'].transform(lambda x: x.shift(1).rolling(5, min_periods=1).mean()))

# Lo mismo pero de 10 partidos
matches_long['winrate_10'] = (matches_long.groupby('Player')['win'].transform(lambda x: x.shift(1).rolling(10, min_periods=1).mean()))

# Calculamos historicos
matches_long["matches_played_before"] = matches_long.groupby("Player").cumcount() # cumcount mira hasta la fila anterior
matches_long["wins_before"] = (matches_long.groupby("Player")["win"].transform(lambda x: x.cumsum().shift(1)).fillna(0)) # cumsum mira hasta la fila actual -> usamos shift
matches_long["win_pct_before"] = (matches_long["wins_before"] / matches_long["matches_played_before"].replace(0, pd.NA))

Sets jugados en ultimos 5 dias y sets jugados en ultimos 30 dias

In [118]:
def sets_ultimos_5_dias(group):
    resultados = []
    # Para cada partido de ese grupo (que tiene los partidos de un jugador)
    for i, (fecha, sets) in enumerate(zip(group['Fecha'], group['setsPartido'])):
        # Calcula la fecha de ese partido - 5 dias
        limite = fecha - pd.Timedelta(days=5)
        # Mira la cantidad de partidos que hubo antes de la fecha del partido pero despues del limite (que es la fecha del partido - 5)
        mask = (group['Fecha'] < fecha) & (group['Fecha'] >= limite)
        # Agarra los partidos que fueron True y suma los sets de esos partidos
        resultados.append(group.loc[mask, 'setsPartido'].sum())
    return pd.Series(resultados, index=group.index)

# Lo mismo que arriba pero cambiando la cantidad de dias
def sets_ultimos_30_dias(group):
    resultados = []
    for i, (fecha, sets) in enumerate(zip(group['Fecha'], group['setsPartido'])):
        limite = fecha - pd.Timedelta(days=30)
        mask = (group['Fecha'] < fecha) & (group['Fecha'] >= limite)
        resultados.append(group.loc[mask, 'setsPartido'].sum())
    return pd.Series(resultados, index=group.index)

matches_long['sets_5d'] = (matches_long.groupby('Player').apply(sets_ultimos_5_dias, include_groups=False).reset_index(level=0, drop=True))
matches_long['sets_30d'] = (matches_long.groupby('Player').apply(sets_ultimos_30_dias, include_groups=False).reset_index(level=0, drop=True))

# Detalle: include_groups = False hace que no se pase la columna Player a la funcion, pero eso no es problema porque esa columna no la usamos

Partidos en ultimos 365 dias

In [119]:
def partidos_ultimos_365_dias(group):
  resultados = []
  # Para cada fecha de los partidos de un jugador:
  for current_match_date in group['Fecha']:
    # Agarramos la fecha del partido - 365 dias
    limite = current_match_date - pd.Timedelta(days=365)
    # Buscamos obtener con True los partidos que cumplen la condicion de haber sido 365 dias antes del partido
    mask = (group['Fecha'] < current_match_date) & (group['Fecha'] >= limite)
    # Contamos la cantidad de filas que cumplen esa condicion
    resultados.append(group.loc[mask].shape[0])
  return pd.Series(resultados, index=group.index)

matches_long['partidos_365d'] = matches_long.groupby('Player').apply(partidos_ultimos_365_dias, include_groups=False).reset_index(level=0, drop=True)

Dias desde el ultimo partido: se asume df ordenado por fecha

In [120]:
def dias_desde_ultimo_partido(group):
    resultados = []
    fechas = group['Fecha'].values
    # Para cada partido del groupby de jugador que entro a la funcion
    for i, current_match_date in enumerate(fechas):
        # Partidos ANTERIORES al partido en cuestion, en terminos de posición en el df, no solo en fecha
        anteriores = group['Fecha'].iloc[:i]

        if anteriores.empty:
            resultados.append(pd.NA)  # Si no hubo partidos antes, ponemos NaN
        else:
            resultados.append((current_match_date - anteriores.max()).days) # Si hubo, ponemos la diferencia entre la fecha el partido en cuestion y la fecha mas reciente a ese partido
    return pd.Series(resultados, index=group.index)

matches_long['dias_ultimo_partido'] = matches_long.groupby('Player').apply(dias_desde_ultimo_partido, include_groups=False).reset_index(level=0, drop=True)

Racha actual de victorias: se asume df ordenado por fecha

In [121]:
def racha_victorias(group):
    resultados = []
    # Para cada fila de un groupby de un jugador
    for i, current_match_date in enumerate(group['Fecha']):
      # Agarramos los partidos jugados antes de ese dia
      partidos_anteriores = group.iloc[:i].sort_values('Fecha', ascending=False)
      racha = 0
      for gano in partidos_anteriores['win']:
          if gano == 1:
              racha += 1
          else:
              break  # Se cortó la racha
      resultados.append(racha)
    return pd.Series(resultados, index=group.index)

matches_long['racha_victorias'] = matches_long.groupby('Player').apply(racha_victorias, include_groups=False).reset_index(level=0, drop=True)

In [122]:
# Podemos hacer una pequeña comprobación de que todo está OK
# matches_long[matches_long['Player'] == 'Nadal R.'].head(20)

Unimos todas las columnas anteriores creadas en matches_long al df original

In [123]:
FEATURE_COLS = ['winrate_5', 'winrate_10', "wins_before", "win_pct_before", 'sets_5d', 'sets_30d', 'partidos_365d', 'dias_ultimo_partido', 'racha_victorias']

# Hacemos un inner join entre el df de matches_long y la columna de jugadores A del df en base al id del partido y el nombre del jugador A
feats_A = (matches_long.merge(df[['Unnamed: 0', 'playerA']], left_on=['Unnamed: 0', 'Player'], right_on=['Unnamed: 0', 'playerA'])
    [['Unnamed: 0'] + FEATURE_COLS].rename(columns={c: f'{c}_A' for c in FEATURE_COLS})) # (aca renombramos las columnas agregandole que son del jugador A)

# Hacemos un inner join entre el df de matches_long y la columna de jugadores B del df en base al id del partido y el nombre del jugador B
feats_B = (matches_long.merge(df[['Unnamed: 0', 'playerB']], left_on=['Unnamed: 0', 'Player'], right_on=['Unnamed: 0', 'playerB'])
    [['Unnamed: 0'] + FEATURE_COLS].rename(columns={c: f'{c}_B' for c in FEATURE_COLS}))

# Mergeamos el df original con un left join con cada df de estadisticas: primero con el de A, despues con el de B
# Como feats_A y feats_B tienen la misma cantidad de filas que el df original (una por cada jugador A o jugador B) -> todo se une bien
df = df.merge(feats_A, on='Unnamed: 0', how='left')
df = df.merge(feats_B, on='Unnamed: 0', how='left')

Hacemos la diferencia entre las variables que acabamos de incorporar

In [124]:
df['diff_winrate_5'] = (df['winrate_5_A'] - df['winrate_5_B'])
df['diff_winrate_10'] = (df['winrate_10_A'] - df['winrate_10_B'])
df['diff_win_pct_before'] = (df['win_pct_before_A'] - df['win_pct_before_B'])
df['diff_sets_5d'] = (df['sets_5d_A'] - df['sets_5d_B'])
df['diff_sets_30d'] = (df['sets_30d_A'] - df['sets_30d_B'])
df['diff_dias_ultimo_partido'] = (df['dias_ultimo_partido_A'] - df['dias_ultimo_partido_B'])
df['diff_partidos_365d'] = (df['partidos_365d_A'] - df['partidos_365d_B'])

Cantidad de partidos previos entre ambos

In [125]:
# Cantidad de partidos previos entre ambos jugadores (Head-to-Head)
# Create a unique identifier for each match pair, sorted alphabetically to handle both playerA vs playerB and playerB vs playerA
df['h2h_pair_id'] = df.apply(lambda row: tuple(sorted([row['playerA'], row['playerB']])), axis=1)

print(f'Esta todo ok? {(df.groupby('h2h_pair_id')['Fecha'].apply(lambda x: x.duplicated().sum()).sum() == 0)}')

# Sort the DataFrame by this pair ID and then by date to ensure correct cumulative counting
df_sorted_h2h = df.sort_values(by=['h2h_pair_id', 'Fecha']).copy()

# Calculate the cumulative count of matches for each unique pair.
# cumcount() starts from 0, so it directly gives the number of previous matches for that pair.
df_sorted_h2h['h2h_matches_previous'] = df_sorted_h2h.groupby('h2h_pair_id').cumcount()

# Merge this new feature back into the original DataFrame using 'Unnamed: 0' (unique match ID)
df = df.merge(df_sorted_h2h[['Unnamed: 0', 'h2h_matches_previous']], on='Unnamed: 0', how='left')

# Drop the temporary column
df.drop(columns=['h2h_pair_id'], inplace=True)

Esta todo ok? True


Otras variables posibles:
- Record en cada superficie, previo al partido

## Encoding de categoricas

Para el nivel del torneo podemos hacer un OrdinalEncoder tranquilamente

In [126]:
df['Series'].value_counts()

,count
Series,
Masters 1000,5884
Grand Slam,5459
ATP500,3954
ATP250,764


In [127]:
from sklearn.preprocessing import OrdinalEncoder # Ordinal Encoder -> con valores de 0 a n cantidad de variables

oe = OrdinalEncoder(categories=[['ATP250', 'ATP500', 'Masters 1000', 'Grand Slam']])
series_oe = oe.fit_transform(df[['Series']])
series_oe = pd.DataFrame(series_oe, columns=['series_level_oe'])
df = pd.concat([df, series_oe], axis=1)

Para Round armamos varias One Hot, porque tenemos ['1st Round', '2nd Round', 'Quarterfinals', 'Semifinals', 'The Final', '3rd Round', '4th Round'] y no sirve hacer un solo OneHot o un LabelEncoder ni otro.

Entonces encodeamos las instancias que son mas importantes: si es o no una final, si es o no una semifinal (donde puede llegar alguno que hizo un batacazo y eso puede influenciar) y si es o no 1st round

In [128]:
df['Round'].value_counts()

,count
Round,
1st Round,7252
2nd Round,4541
3rd Round,1768
Quarterfinals,1096
4th Round,583
Semifinals,548
The Final,273


In [129]:
round_order = {
    '1st Round': 1, '2nd Round': 2, '3rd Round': 3,
    '4th Round': 4, 'Quarterfinals': 5, 'Semifinals': 6, 'The Final': 7
}

df['round_encoded'] = df['Round'].map(round_order)

"Best of" es numerica pero categorica. Le hacemos One Hot

In [130]:
df['Best of'].value_counts()

,count
Best of,
3.0,10612
5.0,5449


In [131]:
df['is_best_of_5'] = (df['Best of'] == 5).astype(int)

Para Surface = ['Hard', 'Clay', 'Grass'] usamos FrequencyEncoder

In [132]:
df['Surface'].value_counts(normalize=True)

,proportion
Surface,
Hard,0.608617
Clay,0.293008
Grass,0.098375


In [133]:
surface_fe = df['Surface'].value_counts(normalize=True) # Generamos un pd.Series con la proporcion de cada valor unicos
df['surface_fe'] = df['Surface'].map(surface_fe)  # Armamos una variable donde cada valor recibe la frecuencia de aparicion de ese valor unico

Para Court = ['Outdoor', 'Indoor'] usamos OneHotEncoder

In [134]:
from sklearn.preprocessing import OneHotEncoder

ohe = OneHotEncoder(sparse_output=False, drop='first')
court_encoded = ohe.fit_transform(df[['Court']])
court_encoded = pd.DataFrame(court_encoded, columns=ohe.get_feature_names_out()).rename(columns={'Court_Outdoor':'is_outdoor'})
df = pd.concat([df, court_encoded], axis=1)

## Limpiamos columnas irrelavantes

In [135]:
columns_drop = ['Unnamed: 0', 'ATP', 'Tournament', 'Date', 'Series', 'Court', 'Surface', 'Round', 'Best of', 'Winner', 'Loser', 'WRank',
       'LRank', 'WPts', 'LPts', 'W1', 'L1', 'W2', 'L2', 'W3', 'L3', 'W4', 'L4', 'W5', 'L5', 'Wsets', 'Lsets', 'Comment', 'B365W',
       'B365L', 'MaxW', 'MaxL', 'AvgW', 'AvgL']

In [136]:
df = df.drop(columns=columns_drop)

In [138]:
# df.to_csv('df_feng_done.csv')